# Milestone 1 EDA — Getting Familiar with VHR Satellite Imagery & GeoTIFFs

**Project:** Cloud Detection in Very-High-Resolution Optical Satellite Imagery — Ursa Space Systems (Break Through Tech AI Studio, Fall 2026)
**Author:** Rushil Dileep
**Milestone:** September — *Foundations & Data* (Issue #2)

This notebook is an individual exploration of the problem space, the data, and the
tooling we'll build on. It is designed to run top-to-bottom in **free Google Colab**.

### What this notebook delivers
1. Loads and displays VHR imagery from the **Maxar Open Data** catalog.
2. Shows several scenes with **different cloud conditions** (deliberately filtered toward higher cloud cover).
3. Demonstrates **windowed / overview reading** of large cloud-optimized GeoTIFFs (COGs).
4. Includes **written observations** on what clouds vs. cloud-like surfaces (smoke, snow, bright rooftops, sand) look like.
5. Summarizes **prior work** and the **evaluation metrics** (IoU, precision/recall, F1).

### Checklist → section map
| Issue checklist item | Section |
|---|---|
| Understand the problem / classical vs. DL tradeoffs | §2 |
| Hands-on with rasterio / pystac / leafmap | §3, §5, §6, §7 |
| Browse catalog, find 2–3 cloudy scenes | §4, §5 |
| Load & display as RGB bands | §6 |
| Tile / window-read large imagery | §7 |
| Observations on clouds vs. cloud-like surfaces | §8 |
| Prior work + metrics | §9 |
| Contribute one scene + one observation | §10 |


## 1. Environment setup

Install the geospatial Python stack and import everything we need. On the first
run in Colab the install takes a minute or two.

In [ ]:
# Install the geospatial stack (run once per Colab session).
!pip install -q rasterio leafmap pystac-client pystac matplotlib numpy

import itertools
import warnings

import numpy as np
import matplotlib.pyplot as plt

import rasterio
from rasterio.windows import Window

import pystac
import leafmap

warnings.filterwarnings("ignore")
print("Libraries imported successfully.")

## 2. The problem: cloud contamination in VHR optical imagery

**What is cloud contamination?** Optical satellites image the Earth in visible
(and sometimes near-infrared) light. Whenever clouds sit between the sensor and the
ground, they **block or scatter that light**, so the pixels record the cloud top
instead of the surface. Thin cloud and haze don't fully block the ground but *wash
it out* — reducing contrast and shifting colors. Either way, the affected pixels are
unusable for measuring what's actually on the ground.

**Why it matters commercially (for Ursa).** Ursa Space Systems turns satellite
imagery into *actionable intelligence* — monitoring economic activity, infrastructure,
and environmental change. Every one of those downstream products assumes it can see
the ground. Cloud-covered pixels silently corrupt those measurements, so being able
to **automatically detect and mask clouds** is a prerequisite for trustworthy
analytics. With VHR imagery (~30–50 cm), a single scene is huge, so manual masking
doesn't scale — we need an automated method.

### Classical signal processing vs. pretrained deep learning
Our project compares two families of approaches:

| | **Classical signal processing** | **Pretrained deep learning** |
|---|---|---|
| Examples | Brightness/saturation thresholding, dark-channel prior, haze indices (HOT) | Zero-shot SAM, pretrained cloud-segmentation CNNs/U-Nets |
| Needs training data? | No | Pretrained already; fine-tuning needs labels |
| Compute | Very light (runs on CPU) | Heavier (benefits from GPU) |
| Interpretability | High — simple, inspectable rules | Lower — learned features |
| Tuning | Manual thresholds, can be brittle across scenes | Generalizes better, but can fail on out-of-distribution scenes |
| Failure modes | Confuses bright non-cloud surfaces (snow, sand, rooftops) | Can mis-handle thin cloud/smoke; domain gap from training data |

The core tension: classical methods are **cheap and transparent but brittle**;
pretrained models are **more robust but opaque and heavier**. The goal for December
is a reproducible, evidence-based comparison — not a single accuracy number.

## 3. Connect to the Maxar Open Data catalog

The [Maxar Open Data Program](https://www.maxar.com/open-data) publishes free VHR
imagery for major events (hurricanes, wildfires, floods, earthquakes). It's exposed
as a **STAC** (SpatioTemporal Asset Catalog).

> **Note:** This is a *static* STAC catalog (plain JSON files on S3), **not** a STAC
> *API*. So we open it with `pystac.Catalog.from_file(...)` and traverse it directly.
> `pystac_client.Client.open(...).search(...)` would **not** work here because a
> static catalog advertises no search conformance — a common gotcha.

In [ ]:
# Open the static Maxar Open Data STAC catalog and list the events (top-level children).
CATALOG_URL = "https://maxar-opendata.s3.amazonaws.com/events/catalog.json"
catalog = pystac.Catalog.from_file(CATALOG_URL)
print(f"Catalog: {catalog.id}")

events = list(catalog.get_children())
print(f"Number of events in catalog: {len(events)}\n")
print("Sample of available events:")
for child in events[:20]:
    print(f"  - {child.id}")

## 4. Pick cloud-prone events (discovered dynamically)

Rather than hardcoding event IDs (which drift and silently break), we **discover**
events whose IDs mention cloud-prone phenomena — hurricanes, wildfires, floods,
storms. If none match, we fall back to the first few events so the notebook always
has something to work with.

In [ ]:
# Keywords for events that tend to have visible cloud / smoke cover.
CLOUD_PRONE_KEYWORDS = ["hurricane", "fire", "flood", "storm", "cyclone", "typhoon", "volcano"]

def find_events(events, keywords, max_events=3):
    """Return event collections whose id matches any keyword (case-insensitive)."""
    matches = [e for e in events if any(k in e.id.lower() for k in keywords)]
    if not matches:
        print("No keyword matches found; falling back to the first events in the catalog.")
        matches = events
    return matches[:max_events]

selected_events = find_events(events, CLOUD_PRONE_KEYWORDS, max_events=3)
print("Selected events:")
for e in selected_events:
    print(f"  - {e.id}")

## 5. Assemble a working set of cloudy scenes

Each event contains many image *items*. Walking an entire event would fire hundreds
of HTTP requests, so we read a **bounded** number of items per event (via
`itertools.islice`), record each item's reported `eo:cloud_cover`, and then pick a
handful of scenes that span **low, medium, and high** cloud cover — exactly the
"filtered toward higher cloud cover" working set the milestone asks for.

In [ ]:
def sample_items(collection, limit=40):
    """Fetch up to `limit` items from an event, with cloud-cover metadata.

    Uses a bounded recursive walk (get_items(recursive=True) + islice) instead of
    the deprecated get_all_items(), to keep the number of HTTP calls reasonable.
    """
    rows = []
    for item in itertools.islice(collection.get_items(recursive=True), limit):
        cc = item.properties.get("eo:cloud_cover")  # may be None
        rows.append({"event": collection.id, "id": item.id, "cloud_cover": cc, "item": item})
    return rows

# Gather candidate scenes across the selected events.
candidates = []
for ev in selected_events:
    rows = sample_items(ev, limit=40)
    candidates.extend(rows)
    print(f"{ev.id}: sampled {len(rows)} items")

# Keep only items that report a cloud-cover value, sorted high -> low.
with_cc = [r for r in candidates if r["cloud_cover"] is not None]
with_cc.sort(key=lambda r: r["cloud_cover"], reverse=True)
print(f"\n{len(with_cc)} / {len(candidates)} candidate items report eo:cloud_cover.")

In [ ]:
def pick_spread(rows, k=3):
    """Pick k scenes spanning the cloud-cover range (high / mid / low)."""
    if not rows:
        return []
    if len(rows) <= k:
        return rows
    idxs = np.linspace(0, len(rows) - 1, k).round().astype(int)
    return [rows[i] for i in idxs]

# Prefer scenes with real cloud-cover metadata; otherwise fall back to any candidates.
pool = with_cc if with_cc else candidates
selected_scenes = pick_spread(pool, k=3)

print("Working set of scenes:")
for s in selected_scenes:
    cc = s["cloud_cover"]
    cc_str = f"{cc:.0f}%" if cc is not None else "N/A"
    print(f"  - [{cc_str:>4} cloud] {s['event']} | {s['id']}")

## 6. Load and display scenes as RGB

VHR scenes are **cloud-optimized GeoTIFFs (COGs)** — often gigabytes. We never need
the full file to look at a scene: COGs store downsampled **overviews**, so we ask
`rasterio` to read the image at a small `out_shape` and it streams only the pixels it
needs. We then apply a per-band **2–98 percentile stretch** so the RGB display has
good contrast.

In [ ]:
def get_visual_href(item):
    """Return the href of the 8-bit RGB 'visual' COG, with sensible fallbacks."""
    for key in ("visual", "rgb", "B3B2B1"):
        if key in item.assets:
            return item.assets[key].href
    # Last resort: first available asset.
    return next(iter(item.assets.values())).href


def read_rgb(href, out_size=512):
    """Read an RGB array (H, W, 3) from a COG at reduced resolution, stretched to [0,1]."""
    with rasterio.open(href) as src:
        n = min(3, src.count)
        bands = list(range(1, n + 1))
        data = src.read(bands, out_shape=(n, out_size, out_size)).astype(float)
    # Percentile stretch each band for display contrast.
    for i in range(data.shape[0]):
        p2, p98 = np.percentile(data[i], (2, 98))
        data[i] = np.clip((data[i] - p2) / (p98 - p2 + 1e-9), 0, 1)
    return np.transpose(data, (1, 2, 0))


# Display the working set side by side.
if selected_scenes:
    n = len(selected_scenes)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    axes = np.atleast_1d(axes)
    for ax, s in zip(axes, selected_scenes):
        try:
            rgb = read_rgb(get_visual_href(s["item"]))
            ax.imshow(rgb)
            cc = s["cloud_cover"]
            cc_str = f"{cc:.0f}% cloud" if cc is not None else "cloud N/A"
            ax.set_title(f"{s['event']}\n{s['id']}\n{cc_str}", fontsize=8)
        except Exception as e:
            ax.set_title(f"Failed to load\n{s['id']}", fontsize=8)
            print(f"Could not load {s['id']}: {e}")
        ax.axis("off")
    plt.suptitle("VHR scenes from Maxar Open Data (different cloud conditions)", y=1.03)
    plt.tight_layout()
    plt.show()
else:
    print("No scenes selected — re-run the selection cells above.")

## 7. Windowed reading — working with full-resolution tiles

The display above used overviews. To work at **native resolution** without loading
a multi-GB file, we read a single **window** (a tile) from the full-res COG. This is
the pattern we'll reuse later for tiling scenes into patches for cloud detection.

In [ ]:
if selected_scenes:
    href = get_visual_href(selected_scenes[0]["item"])
    with rasterio.open(href) as src:
        print(f"Full-res dimensions: {src.width} x {src.height}, bands: {src.count}")
        print(f"CRS: {src.crs} | cloud-optimized: {src.is_cloud_optimized}")

        # Read a 1024x1024 tile from the center of the scene at native resolution.
        tile = 1024
        col_off = max(0, (src.width - tile) // 2)
        row_off = max(0, (src.height - tile) // 2)
        win = Window(col_off, row_off,
                     min(tile, src.width), min(tile, src.height))
        patch = src.read([1, 2, 3], window=win).astype(float)

    for i in range(patch.shape[0]):
        p2, p98 = np.percentile(patch[i], (2, 98))
        patch[i] = np.clip((patch[i] - p2) / (p98 - p2 + 1e-9), 0, 1)

    plt.figure(figsize=(6, 6))
    plt.imshow(np.transpose(patch, (1, 2, 0)))
    plt.title(f"Native-resolution {tile}x{tile} window\n{selected_scenes[0]['id']}", fontsize=9)
    plt.axis("off")
    plt.show()
    print(f"Window array shape (bands, H, W): {patch.shape}")
else:
    print("No scene available for the windowed-read demo.")

## 8. Interactive exploration with leafmap

`leafmap` can stream a COG directly onto an interactive map, so we can pan/zoom over
a scene and see the clouds in geographic context. (If the map doesn't render in your
environment, the static displays above still cover the deliverable.)

In [ ]:
if selected_scenes:
    href = get_visual_href(selected_scenes[0]["item"])
    m = leafmap.Map()
    try:
        m.add_cog_layer(href, name=selected_scenes[0]["event"])
    except Exception as e:
        print(f"Could not add COG layer interactively: {e}")
    m
else:
    print("No scene available for the interactive map.")

## 9. Observations — clouds vs. cloud-like surfaces

These are the visual cues for telling clouds apart from things that *look* like clouds.
This is the intuition our classical thresholds and model comparisons will have to respect.

**Clouds**
- **Thick cloud:** very bright, near-white, usually **desaturated** (R≈G≈B). Soft, billowy,
  irregular edges. Opaque — the ground underneath is completely gone. Often casts a
  **displaced dark shadow** on the surface, offset by sun angle.
- **Thin cloud / haze:** semi-transparent, *low-contrast* veil. The ground is still
  faintly visible but washed out and bluish/gray. This is the **hardest** case — a hard
  brightness threshold either misses it or over-masks. Haze indices (like HOT) target
  exactly this regime.

**Cloud-like surfaces (false positives to watch for)**
- **Smoke (wildfires):** wispy, **directional** plumes streaming from a source, often
  **brown/gray/bluish** rather than cloud-white, and semi-transparent. Follows wind,
  not the billowy cloud shape. A brightness-only method may miss it (too dark); color
  and texture cues help.
- **Snow / ice:** bright and desaturated like cloud, but it **conforms to the terrain**
  (valleys, ridgelines, field boundaries), has **sharp, ground-registered edges**, and
  casts **no displaced shadow**. The classic failure case for brightness thresholds.
- **Bright rooftops / concrete / solar farms:** **geometric**, small, sharp-edged, and
  often **higher-saturation** (not neutral white). Spatially tied to urban layout.
- **Sand / desert / dry lakebeds:** bright but **warm-toned** (yellow/tan), textured,
  and part of the continuous ground surface — no shadow, no billowy structure.

**Most useful discriminators:** (1) **saturation** — true cloud is near-neutral;
(2) **cast shadow** — a displaced dark companion strongly implies real cloud;
(3) **edge/texture** — soft billowy vs. sharp geometric/terrain-locked;
(4) **color tone** — white vs. brown (smoke) / warm (sand).

## 10. Prior work and evaluation metrics

### Cloud-detection methods
- **HOT (Haze-Optimized Transform):** a visible-band index exploiting that clear-sky
  pixels follow a tight blue–red correlation ("clear-sky line"); haze/thin cloud
  deviates from it. Good at the thin-cloud/haze regime that brightness thresholds miss.
- **Dark-channel prior:** originally a dehazing method. In a haze-free patch, at least
  one RGB channel has very low intensity somewhere locally; haze/cloud raises that
  minimum. The per-pixel dark channel becomes a cheap haze/cloud indicator — a strong
  classical baseline for our October milestone.
- **Brightness / saturation thresholding:** the simplest baseline — clouds are bright
  and low-saturation. Fast and interpretable, but brittle against snow/sand/rooftops.
- **SAM (Segment Anything Model) for remote sensing:** a promptable foundation
  segmentation model applied **zero-shot**. It produces clean masks but is class-agnostic
  (it segments *regions*, not "cloud" specifically), so it needs prompting/post-labeling.
  Represents the pretrained-DL arm of our comparison.

### Evaluation metrics (cloud vs. non-cloud pixels)
Cloud masking is **binary pixel segmentation**, usually **class-imbalanced**, so plain
accuracy is misleading. Standard metrics:
- **IoU (Jaccard):** `|pred ∩ truth| / |pred ∪ truth|` — the primary segmentation metric;
  penalizes both missed cloud and over-masking.
- **Precision:** of the pixels we called cloud, how many really are (controls over-masking).
- **Recall:** of the true cloud pixels, how many we caught (controls missed cloud).
- **F1 (Dice):** harmonic mean of precision and recall — a single balanced score.

Because there are **no official ground-truth masks**, our success criterion is a
*reproducible comparison* against a small student-curated reference set, plus a
**qualitative error analysis** of where each method breaks (thin cloud, smoke, bright
surfaces).

## 11. My contribution & open questions

**Contributed scene:** the highest-cloud-cover scene selected in §5 (printed above).

**Contributed observation:** the single most reliable cue I found for separating true
cloud from snow and bright rooftops is the **displaced cast shadow** combined with
**low saturation** — snow is bright but terrain-locked with no offset shadow, and
rooftops are bright but geometric and more saturated. Thin cloud/haze remains the
hardest case and is where I expect the classical brightness baseline to struggle most.

**Open questions for the team**
1. Should our curated reference set deliberately over-sample hard cases (thin cloud,
   smoke, snow) so the October metrics are meaningful?
2. Do we standardize on the 8-bit `visual` asset, or bring in the multispectral/NIR
   product for the classical baselines (stretch goal)?
3. What tile size and overlap do we want for patch-based processing?
